# Prédire le vainqueur d'une Coupe du Monde de football avec Python

Notebook de l'article [Prédire le vainqueur d'une Coupe du Monde de football avec Python](https://moncoachdata.com/blog/predire-le-vainqueur-de-la-coupe-du-monde/) (MonCoachData).

On utilise les matchs des Coupes du Monde de 1930 à 2018 pour estimer la force de chaque équipe, puis la loi de Poisson pour prédire les matchs, la phase de poules et le tableau final de la Coupe du Monde 2022. Exécute les cellules **dans l'ordre** : les deux fichiers CSV sont lus directement depuis le dépôt GitHub `moncoachdata/blog-data-science` (il faut une connexion internet).

Si tu travailles sur ton ordinateur, installe d'abord les bibliothèques (déjà présentes dans Google Colab) : `pip install pandas scipy numpy`.

## 1. Charger les données

Attendu : `(900, 5) (64, 4)`.

In [1]:
import pandas as pd

BASE = "https://raw.githubusercontent.com/moncoachdata/blog-data-science/main/predire-le-vainqueur-de-la-coupe-du-monde/"

df_historical_data = pd.read_csv(BASE + "clean_fifa_worldcup_matches.csv")
df_fixture = pd.read_csv(BASE + "clean_fifa_worldcup_fixture.csv")
print(df_historical_data.shape, df_fixture.shape)
df_historical_data.tail(4)

(900, 5) (64, 4)


,HomeTeam,AwayTeam,Year,HomeGoals,AwayGoals
896,France,Belgium,2018,1,0
897,Croatia,England,2018,2,1
898,Belgium,England,2018,2,0
899,France,Croatia,2018,4,2


## 2. Les buts suivent-ils une loi de Poisson ?

Moyenne attendue : 1,42 but par équipe et par match.

In [2]:
import numpy as np
from scipy.stats import poisson

buts = pd.concat([df_historical_data["HomeGoals"], df_historical_data["AwayGoals"]])
lamb = buts.mean()
print("Moyenne de buts par équipe et par match :", round(lamb, 2))

reel = buts.value_counts(normalize=True).sort_index()
theorie = pd.Series(poisson.pmf(reel.index, lamb), index=reel.index)
pd.DataFrame({"réel": reel, "Poisson": theorie}).round(3).head(7)

Moyenne de buts par équipe et par match : 1.42


,réel,Poisson
0,0.284,0.243
1,0.334,0.344
2,0.208,0.243
3,0.098,0.115
4,0.042,0.041
5,0.014,0.011
6,0.010,0.003


## 3. Calculer la force de chaque équipe

Attendu : 85 équipes.

In [3]:
def get_team_strength(df_matches):
    """Moyenne de buts marqués et encaissés par équipe, domicile et extérieur confondus."""
    cols = ["Team", "GoalsScored", "GoalsConceded"]
    home = df_matches.rename(columns={"HomeTeam": "Team", "HomeGoals": "GoalsScored", "AwayGoals": "GoalsConceded"})
    away = df_matches.rename(columns={"AwayTeam": "Team", "AwayGoals": "GoalsScored", "HomeGoals": "GoalsConceded"})
    return pd.concat([home[cols], away[cols]]).groupby("Team").mean()


df_team_strength = get_team_strength(df_historical_data)
print(df_team_strength.shape)
favoris = ["Brazil", "Germany", "France", "Argentina", "Spain", "England", "Netherlands", "United States"]
df_team_strength.loc[favoris].round(2)

(85, 2)


,GoalsScored,GoalsConceded
Team,,
Brazil,2.10,0.96
Germany,2.02,1.02
France,1.82,1.17
Argentina,1.69,1.15
Spain,1.57,1.14
England,1.32,0.93
Netherlands,1.72,0.96
United States,1.12,1.88


In [4]:
lamb_england = df_team_strength.at["England", "GoalsScored"] * df_team_strength.at["United States", "GoalsConceded"]
lamb_usa = df_team_strength.at["United States", "GoalsScored"] * df_team_strength.at["England", "GoalsConceded"]
print("Buts attendus : Angleterre", round(lamb_england, 2), "- États-Unis", round(lamb_usa, 2))
print("Le Qatar est dans le tableau ?", "Qatar" in df_team_strength.index)

Buts attendus : Angleterre 2.48 - États-Unis 1.04
Le Qatar est dans le tableau ? False


## 4. Prédire un match : `predict_points`

Attendu pour Angleterre – États-Unis : 2,24 points contre 0,59.

In [5]:
from scipy.stats import poisson


def predict_points(home, away, strength):
    if home not in strength.index or away not in strength.index:
        return (0, 0)  # équipe sans historique en Coupe du monde
    lamb_home = strength.at[home, "GoalsScored"] * strength.at[away, "GoalsConceded"]
    lamb_away = strength.at[away, "GoalsScored"] * strength.at[home, "GoalsConceded"]
    prob_home, prob_draw, prob_away = 0, 0, 0
    for x in range(0, 11):  # buts de l'équipe 1
        for y in range(0, 11):  # buts de l'équipe 2
            p = poisson.pmf(x, lamb_home) * poisson.pmf(y, lamb_away)
            if x == y:
                prob_draw += p
            elif x > y:
                prob_home += p
            else:
                prob_away += p
    points_home = 3 * prob_home + prob_draw
    points_away = 3 * prob_away + prob_draw
    return (points_home, points_away)

In [6]:
points_england, points_usa = predict_points("England", "United States", df_team_strength)
print("Angleterre :", round(points_england, 2), "points - États-Unis :", round(points_usa, 2))

Angleterre : 2.24 points - États-Unis : 0.59


## 5. Quel est le score le plus probable ?

In [7]:
def score_probabilities(home, away, strength, max_goals=10):
    lamb_home = strength.at[home, "GoalsScored"] * strength.at[away, "GoalsConceded"]
    lamb_away = strength.at[away, "GoalsScored"] * strength.at[home, "GoalsConceded"]
    goals = np.arange(max_goals + 1)
    matrix = np.outer(poisson.pmf(goals, lamb_home), poisson.pmf(goals, lamb_away))
    return pd.DataFrame(matrix, index=goals, columns=goals)  # lignes : buts de l'Angleterre


proba = score_probabilities("England", "United States", df_team_strength)
print("Somme des probabilités :", round(proba.to_numpy().sum(), 4))
for (x, y), p in proba.stack().sort_values(ascending=False).head(3).items():
    print(f"Angleterre {x} - {y} États-Unis : {p:.1%}")

Somme des probabilités : 0.9999
Angleterre 2 - 1 États-Unis : 9.5%
Angleterre 2 - 0 États-Unis : 9.1%
Angleterre 3 - 1 États-Unis : 7.8%


## 6. Prédire la phase de poules

In [8]:
group_matches = df_fixture.iloc[:48].copy()  # les 48 matchs de poule (8 groupes x 6 matchs)
group_matches["group"] = np.repeat(list("ABCDEFGH"), 6)

group_matches[["points_home", "points_away"]] = [
    predict_points(home, away, df_team_strength)
    for home, away in zip(group_matches["home"], group_matches["away"])
]
group_matches.loc[group_matches["group"] == "B", ["home", "away", "points_home", "points_away"]].round(2)

,home,away,points_home,points_away
6,England,Iran,2.39,0.43
7,United States,Wales,0.94,1.80
8,Wales,Iran,1.99,0.71
9,England,United States,2.24,0.59
10,Wales,England,1.08,1.59
11,Iran,United States,0.95,1.82


In [9]:
home_pts = group_matches[["group", "home", "points_home"]].set_axis(["group", "team", "points"], axis=1)
away_pts = group_matches[["group", "away", "points_away"]].set_axis(["group", "team", "points"], axis=1)

df_groups = (
    pd.concat([home_pts, away_pts])
    .groupby(["group", "team"], as_index=False)["points"].sum()
    .sort_values(["group", "points"], ascending=[True, False])
    .reset_index(drop=True)
)
df_groups["rank"] = df_groups.groupby("group").cumcount() + 1
df_groups[df_groups["group"].isin(["A", "B"])].round(2)

,group,team,points,rank
0,A,Netherlands,3.89,1
1,A,Senegal,2.45,2
2,A,Ecuador,1.98,3
3,A,Qatar,0.00,4
4,B,England,6.22,1
5,B,Wales,4.88,2
6,B,United States,3.35,3
7,B,Iran,2.09,4


In [10]:
winners = df_groups[df_groups["rank"] == 1].set_index("group")["team"]
runners_up = df_groups[df_groups["rank"] == 2].set_index("group")["team"]

for group in winners.index:
    print(f"Groupe {group} : 1er {winners[group]}, 2e {runners_up[group]}")

Groupe A : 1er Netherlands, 2e Senegal
Groupe B : 1er England, 2e Wales
Groupe C : 1er Argentina, 2e Poland
Groupe D : 1er France, 2e Denmark
Groupe E : 1er Germany, 2e Spain
Groupe F : 1er Croatia, 2e Belgium
Groupe G : 1er Brazil, 2e Switzerland
Groupe H : 1er Portugal, 2e Uruguay


## 7. Comparer avec la réalité de 2022

Attendu : 11 qualifiés sur 16 bien prédits.

In [11]:
real_round_of_16 = {
    "Netherlands", "United States", "Argentina", "Australia", "France", "Poland", "England", "Senegal",
    "Japan", "Croatia", "Brazil", "South Korea", "Morocco", "Spain", "Portugal", "Switzerland",
}
predicted_round_of_16 = set(winners.tolist() + runners_up.tolist())
print(len(predicted_round_of_16 & real_round_of_16), "équipes qualifiées sur 16 bien prédites")
print("Prédites à tort :", sorted(predicted_round_of_16 - real_round_of_16))
print("Oubliées :", sorted(real_round_of_16 - predicted_round_of_16))

11 équipes qualifiées sur 16 bien prédites
Prédites à tort : ['Belgium', 'Denmark', 'Germany', 'Uruguay', 'Wales']
Oubliées : ['Australia', 'Japan', 'Morocco', 'South Korea', 'United States']


## 8. Prédire la phase éliminatoire

Attendu : le Brésil gagne la finale contre la France.

In [12]:
knockout = df_fixture.iloc[48:].copy()  # matchs 49 à 64

placeholders = {}
for group in winners.index:
    placeholders[f"Winners Group {group}"] = winners[group]
    placeholders[f"Runners-up Group {group}"] = runners_up[group]

knockout[["home", "away"]] = knockout[["home", "away"]].replace(placeholders)
knockout.iloc[:8]

,home,score,away,year
48,Netherlands,Match 49,Wales,2022
49,Argentina,Match 50,Denmark,2022
50,France,Match 52,Poland,2022
51,England,Match 51,Senegal,2022
52,Germany,Match 53,Belgium,2022
53,Brazil,Match 54,Uruguay,2022
54,Croatia,Match 55,Spain,2022
55,Portugal,Match 56,Switzerland,2022


In [13]:
def get_winner(matches, strength):
    matches = matches.copy()
    winners = []
    for _, row in matches.iterrows():
        points_home, points_away = predict_points(row["home"], row["away"], strength)
        winners.append(row["home"] if points_home > points_away else row["away"])
    matches["winner"] = winners
    return matches


round_of_16 = get_winner(knockout.iloc[:8], df_team_strength)
round_of_16[["home", "away", "winner"]]

,home,away,winner
48,Netherlands,Wales,Netherlands
49,Argentina,Denmark,Argentina
50,France,Poland,France
51,England,Senegal,England
52,Germany,Belgium,Germany
53,Brazil,Uruguay,Brazil
54,Croatia,Spain,Spain
55,Portugal,Switzerland,Portugal


In [14]:
def next_round(previous_round, next_matches, strength):
    """Remplace « Winners Match N » par le vainqueur du match N, puis prédit les vainqueurs."""
    qualified = dict(zip("Winners " + previous_round["score"], previous_round["winner"]))
    next_matches = next_matches.copy()
    next_matches[["home", "away"]] = next_matches[["home", "away"]].replace(qualified)
    return get_winner(next_matches, strength)


quarter_finals = next_round(round_of_16, knockout.iloc[8:12], df_team_strength)
semi_finals = next_round(quarter_finals, knockout.iloc[12:14], df_team_strength)
final = next_round(semi_finals, knockout.iloc[[15]], df_team_strength)

for name, df_round in [("Quarts", quarter_finals), ("Demi-finales", semi_finals), ("Finale", final)]:
    print(name)
    for _, row in df_round.iterrows():
        print("  ", row["home"], "-", row["away"], "->", row["winner"])

Quarts
   Germany - Brazil -> Brazil
   Netherlands - Argentina -> Netherlands
   Spain - Portugal -> Portugal
   England - France -> France
Demi-finales
   Netherlands - Brazil -> Brazil
   France - Portugal -> France
Finale
   Brazil - France -> Brazil


## 9. Tester le modèle sur 1998-2018

Attendu : 163 bons pronostics sur 231 matchs (70,6 %).

In [15]:
results = []
for year in [1998, 2002, 2006, 2010, 2014, 2018]:
    strength = get_team_strength(df_historical_data[df_historical_data["Year"] < year])  # uniquement le passé
    edition = df_historical_data[df_historical_data["Year"] == year]
    good, total = 0, 0
    for _, m in edition.iterrows():
        if m["HomeGoals"] == m["AwayGoals"]:
            continue  # on ne garde que les matchs avec un vainqueur
        points_home, points_away = predict_points(m["HomeTeam"], m["AwayTeam"], strength)
        if points_home == points_away:
            continue  # équipe sans historique : pas de prédiction
        winner_is_home = m["HomeGoals"] > m["AwayGoals"]
        good += (points_home > points_away) == winner_is_home
        total += 1
    results.append({"Année": year, "Matchs évalués": total, "Bons pronostics": int(good), "Réussite": round(100 * good / total, 1)})

df_backtest = pd.DataFrame(results)
print(df_backtest.to_string(index=False))
print("Total :", df_backtest["Bons pronostics"].sum(), "/", df_backtest["Matchs évalués"].sum(),
      "=", round(100 * df_backtest["Bons pronostics"].sum() / df_backtest["Matchs évalués"].sum(), 1), "%")

 Année  Matchs évalués  Bons pronostics  Réussite
  1998              31               21      67.7
  2002              36               25      69.4
  2006              28               22      78.6
  2010              42               31      73.8
  2014              48               33      68.8
  2018              46               31      67.4
Total : 163 / 231 = 70.6 %


## Tout le code

In [16]:
import numpy as np
import pandas as pd
from scipy.stats import poisson

BASE = "https://raw.githubusercontent.com/moncoachdata/blog-data-science/main/predire-le-vainqueur-de-la-coupe-du-monde/"
df_historical_data = pd.read_csv(BASE + "clean_fifa_worldcup_matches.csv")
df_fixture = pd.read_csv(BASE + "clean_fifa_worldcup_fixture.csv")


def get_team_strength(df_matches):
    """Moyenne de buts marqués et encaissés par équipe, domicile et extérieur confondus."""
    cols = ["Team", "GoalsScored", "GoalsConceded"]
    home = df_matches.rename(columns={"HomeTeam": "Team", "HomeGoals": "GoalsScored", "AwayGoals": "GoalsConceded"})
    away = df_matches.rename(columns={"AwayTeam": "Team", "AwayGoals": "GoalsScored", "HomeGoals": "GoalsConceded"})
    return pd.concat([home[cols], away[cols]]).groupby("Team").mean()


def predict_points(home, away, strength):
    if home not in strength.index or away not in strength.index:
        return (0, 0)  # équipe sans historique en Coupe du monde
    lamb_home = strength.at[home, "GoalsScored"] * strength.at[away, "GoalsConceded"]
    lamb_away = strength.at[away, "GoalsScored"] * strength.at[home, "GoalsConceded"]
    prob_home, prob_draw, prob_away = 0, 0, 0
    for x in range(0, 11):
        for y in range(0, 11):
            p = poisson.pmf(x, lamb_home) * poisson.pmf(y, lamb_away)
            if x == y:
                prob_draw += p
            elif x > y:
                prob_home += p
            else:
                prob_away += p
    return (3 * prob_home + prob_draw, 3 * prob_away + prob_draw)


def get_winner(matches, strength):
    matches = matches.copy()
    winners = []
    for _, row in matches.iterrows():
        points_home, points_away = predict_points(row["home"], row["away"], strength)
        winners.append(row["home"] if points_home > points_away else row["away"])
    matches["winner"] = winners
    return matches


def next_round(previous_round, next_matches, strength):
    qualified = dict(zip("Winners " + previous_round["score"], previous_round["winner"]))
    next_matches = next_matches.copy()
    next_matches[["home", "away"]] = next_matches[["home", "away"]].replace(qualified)
    return get_winner(next_matches, strength)


# 1. Force de chaque équipe
df_team_strength = get_team_strength(df_historical_data)

# 2. Phase de poule : points espérés
group_matches = df_fixture.iloc[:48].copy()
group_matches["group"] = np.repeat(list("ABCDEFGH"), 6)
group_matches[["points_home", "points_away"]] = [
    predict_points(home, away, df_team_strength)
    for home, away in zip(group_matches["home"], group_matches["away"])
]
home_pts = group_matches[["group", "home", "points_home"]].set_axis(["group", "team", "points"], axis=1)
away_pts = group_matches[["group", "away", "points_away"]].set_axis(["group", "team", "points"], axis=1)
df_groups = (
    pd.concat([home_pts, away_pts])
    .groupby(["group", "team"], as_index=False)["points"].sum()
    .sort_values(["group", "points"], ascending=[True, False])
    .reset_index(drop=True)
)
df_groups["rank"] = df_groups.groupby("group").cumcount() + 1
winners = df_groups[df_groups["rank"] == 1].set_index("group")["team"]
runners_up = df_groups[df_groups["rank"] == 2].set_index("group")["team"]

# 3. Phase éliminatoire
knockout = df_fixture.iloc[48:].copy()
placeholders = {}
for group in winners.index:
    placeholders[f"Winners Group {group}"] = winners[group]
    placeholders[f"Runners-up Group {group}"] = runners_up[group]
knockout[["home", "away"]] = knockout[["home", "away"]].replace(placeholders)

round_of_16 = get_winner(knockout.iloc[:8], df_team_strength)
quarter_finals = next_round(round_of_16, knockout.iloc[8:12], df_team_strength)
semi_finals = next_round(quarter_finals, knockout.iloc[12:14], df_team_strength)
final = next_round(semi_finals, knockout.iloc[[15]], df_team_strength)

print("Vainqueur prédit :", final.loc[final.index[0], "winner"])

Vainqueur prédit : Brazil
